In [91]:
import os

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_DISABLE_FAST_FORWARD"] = "1"
os.environ["UNSLOTH_DISABLE_PATCHING"] = "1"

In [92]:
import unsloth

from unsloth import FastVisionModel

In [93]:
import os
import tarfile
import shutil
import pandas as pd
import torch

from datetime import datetime
from PIL import Image
from datasets import Dataset

In [94]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GB10


In [95]:
torch.manual_seed(3407)
torch.cuda.manual_seed_all(3407)

In [96]:
GLOBAL_RULES = """
Atsakyk trumpai lietuvių kalba.
Pateik tik medicininį atsakymą.
Nenaudok paaiškinimų.
"""

In [175]:
timestamp = datetime.now().strftime(
    "%Y%m%d_%H"
)

EXPERIMENT_NAME = (
    f"pathvqa_train_"
    f"other_test_{timestamp}"
)

In [ ]:
DATASETS = {

    "pathvqa": {

        "csv_train": "path-vqa-train.csv",

        "csv_val": "path-vqa-validate.csv",

        "csv_test": "path-vqa-test.csv",

        "tar_files": [
            "pathvqa.tar"
        ],
    },

    "vqarad": {

        "csv_train": "vqa-rad-train.csv",

        "csv_val": "vqa-rad-validate.csv",

        "csv_test": "vqa-rad-test.csv",

        "tar_files": [
            "vqarad.tar",
        ],
    },

    "slake": {

        "csv_train": "slake-train.csv",

        "csv_val": "slake-validate.csv",

        "csv_test": "slake-test.csv",

        "tar_files": [
            "slake.tar",
        ],
    },
}

TRAIN_DATASETS = [
    "pathvqa",
]

VAL_DATASETS = [
    "pathvqa",
]

TEST_DATASETS = [
    "slake",
    "vqarad",
]

In [177]:
BASE_DIR = os.path.abspath("..")

TAR_DIR = os.path.join(BASE_DIR, "tar")
CSV_DIR = os.path.join(BASE_DIR, "csv")

MEDGEMMA_DIR = os.path.join(BASE_DIR, "medgemma")

CHECKPOINT_ROOT = os.path.join(
    MEDGEMMA_DIR,
    "checkpoints"
)

EXPERIMENT_ROOT = os.path.join(
    MEDGEMMA_DIR,
    "experiments"
)

EXPERIMENT_DIR = os.path.join(
    EXPERIMENT_ROOT,
    EXPERIMENT_NAME
)

CHECKPOINT_DIR = os.path.join(
    CHECKPOINT_ROOT,
    EXPERIMENT_NAME
)

PREDICTIONS_DIR = os.path.join(
    EXPERIMENT_DIR,
    "predictions"
)

LOG_DIR = os.path.join(
    EXPERIMENT_DIR,
    "tensorboard"
)

BEST_MODEL_DIR = os.path.join(
    EXPERIMENT_DIR,
    "best_model"
)

LOCAL_IMAGE_ROOT = os.path.join(
    BASE_DIR,
    "temp_images"
)

In [178]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(LOCAL_IMAGE_ROOT, exist_ok=True)

In [179]:
ALL_REQUIRED_DATASETS = list(set(
    TRAIN_DATASETS +
    VAL_DATASETS +
    TEST_DATASETS
))

print("\nDATASETS REQUIRED:")
print(ALL_REQUIRED_DATASETS)

for dataset_name in ALL_REQUIRED_DATASETS:

    dataset_extract_path = os.path.join(
        LOCAL_IMAGE_ROOT,
        dataset_name
    )

    if os.path.exists(dataset_extract_path):

        print(f"\nSKIPPING EXTRACTION: {dataset_name}")
        continue

    dataset_tar_dir = os.path.join(
        TAR_DIR,
        dataset_name
    )

    tar_files = DATASETS[
        dataset_name
    ]["tar_files"]

    print(f"\nEXTRACTING DATASET: {dataset_name}")

    for tar_name in tar_files:

        tar_path = os.path.join(
            dataset_tar_dir,
            tar_name
        )

        print("EXTRACTING:", tar_name)

        with tarfile.open(tar_path, "r") as tar:

            tar.extractall(LOCAL_IMAGE_ROOT)

print("\nALL REQUIRED DATASETS READY.")


DATASETS REQUIRED:
['vqarad', 'slake', 'pathvqa']

SKIPPING EXTRACTION: vqarad

SKIPPING EXTRACTION: slake

SKIPPING EXTRACTION: pathvqa

ALL REQUIRED DATASETS READY.


In [180]:
from datasets import Dataset


def load_split(dataset_names, split):

    dfs = []

    for dataset_name in dataset_names:

        if split == "train":
            csv_name = DATASETS[dataset_name]["csv_train"]

        elif split == "val":
            csv_name = DATASETS[dataset_name]["csv_val"]

        elif split == "test":
            csv_name = DATASETS[dataset_name]["csv_test"]

        else:
            raise Exception(f"INVALID SPLIT: {split}")

        csv_path = os.path.join(
            CSV_DIR,
            dataset_name,
            csv_name
        )

        print("\nLOADING CSV:")
        print(csv_path)

        df = pd.read_csv(
            csv_path,
            encoding="utf-8-sig"
        )

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
        )

        required_columns = [
            "image",
            "question_lt",
            "answer_lt",
        ]

        for col in required_columns:

            if col not in df.columns:
                raise Exception(f"MISSING COLUMN: {col}")

        df["dataset"] = dataset_name

        dfs.append(df)

    combined_df = pd.concat(
        dfs,
        ignore_index=True
    )

    return combined_df

In [181]:
from datasets import Dataset


def load_split(dataset_names, split):

    dfs = []

    for dataset_name in dataset_names:

        if split == "train":
            csv_name = DATASETS[dataset_name]["csv_train"]

        elif split == "val":
            csv_name = DATASETS[dataset_name]["csv_val"]

        elif split == "test":
            csv_name = DATASETS[dataset_name]["csv_test"]

        else:
            raise Exception(f"INVALID SPLIT: {split}")

        csv_path = os.path.join(
            CSV_DIR,
            dataset_name,
            csv_name
        )

        print("\nLOADING CSV:")
        print(csv_path)

        df = pd.read_csv(
            csv_path,
            encoding="utf-8-sig"
        )

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
        )

        required_columns = [
            "image",
            "question_lt",
            "answer_lt",
        ]

        for col in required_columns:

            if col not in df.columns:
                raise Exception(f"MISSING COLUMN: {col}")

        df["dataset"] = dataset_name

        dfs.append(df)

    combined_df = pd.concat(
        dfs,
        ignore_index=True
    )

    return combined_df

In [ ]:

train_df = load_split(
    TRAIN_DATASETS,
    "train"
)

val_df = load_split(
    VAL_DATASETS,
    "val"
)

test_df = load_split(
    TEST_DATASETS,
    "test"
)

print("\n===================================")

print("TRAIN SAMPLES:")
print(len(train_df))

print("\nVAL SAMPLES:")
print(len(val_df))

print("\nTEST SAMPLES:")
print(len(test_df))

print("\n===================================")

print("\nTRAIN HEAD:")
print(train_df.head())

print("\nVAL HEAD:")
print(val_df.head())

print("\nTEST HEAD:")
print(test_df.head())


LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/pathvqa/path-vqa-train.csv

LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/pathvqa/path-vqa-validate.csv

LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/slake/slake-test.csv

LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/vqarad/vqa-rad-test.csv

TRAIN SAMPLES:
19654

VAL SAMPLES:
6259

TEST SAMPLES:
1287


TRAIN HEAD:
                             image  \
0  output_train_0_images_img_0.jpg   
1  output_train_0_images_img_1.jpg   
2  output_train_0_images_img_2.jpg   
3  output_train_0_images_img_3.jpg   
4  output_train_0_images_img_4.jpg   

                                         question_lt  \
0  Kur yra kepenų kamieninės ląstelės (ovalios lą...   
1  Kas čia dažoma imunohistochemine citokeratino ...   
2         Ką reiškia baltų kreidinių nuosėdų plotai?   
3  Ar embolija gaunama iš apatinių galūnių giliųj...   
4         Kaip apibūdinama hiperplazija be atipijos?   

                         

In [ ]:

train_df = load_split(
    TRAIN_DATASETS,
    "train"
)

val_df_full = load_split(
    VAL_DATASETS,
    "val"
)

VAL_SAMPLE_SIZE = 1000   # change to 500, 1000, 2000, etc.

if len(val_df_full) > VAL_SAMPLE_SIZE:
    val_df = val_df_full.sample(
        n=VAL_SAMPLE_SIZE,
        random_state=3407
    ).reset_index(drop=True)
else:
    val_df = val_df_full.reset_index(drop=True)

test_df = load_split(
    TEST_DATASETS,
    "test"
)

print("\n===================================")

print("TRAIN SAMPLES:")
print(len(train_df))

print("\nVAL SAMPLES USED:")
print(len(val_df))

print("\nVAL SAMPLES FULL:")
print(len(val_df_full))

print("\nTEST SAMPLES:")
print(len(test_df))

print("\n===================================")

print("\nTRAIN HEAD:")
print(train_df.head())

print("\nVAL HEAD:")
print(val_df.head())

print("\nTEST HEAD:")
print(test_df.head())




LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/pathvqa/path-vqa-train.csv

LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/pathvqa/path-vqa-validate.csv

LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/slake/slake-test.csv

LOADING CSV:
/home/vml/notebooks/test/med-lt-project/csv/vqarad/vqa-rad-test.csv

TRAIN SAMPLES:
19654

VAL SAMPLES USED:
1000

VAL SAMPLES FULL:
6259

TEST SAMPLES:
1287


TRAIN HEAD:
                             image  \
0  output_train_0_images_img_0.jpg   
1  output_train_0_images_img_1.jpg   
2  output_train_0_images_img_2.jpg   
3  output_train_0_images_img_3.jpg   
4  output_train_0_images_img_4.jpg   

                                         question_lt  \
0  Kur yra kepenų kamieninės ląstelės (ovalios lą...   
1  Kas čia dažoma imunohistochemine citokeratino ...   
2         Ką reiškia baltų kreidinių nuosėdų plotai?   
3  Ar embolija gaunama iš apatinių galūnių giliųj...   
4         Kaip apibūdinama hiperplazija be atipijos? 

In [184]:
missing_files = []

for df in [train_df, val_df, test_df]:

    for _, row in df.iterrows():

        image_path = os.path.join(
            LOCAL_IMAGE_ROOT,
            row["dataset"],
            row["image"]
        )

        if not os.path.exists(image_path):
            missing_files.append(image_path)

print("\nMISSING FILES:", len(missing_files))

if len(missing_files) > 0:

    print("\nFIRST 20 MISSING:")

    for p in missing_files[:20]:
        print(p)

    raise Exception("Missing images detected.")

else:
    print("\nALL IMAGES VERIFIED SUCCESSFULLY")


MISSING FILES: 0

ALL IMAGES VERIFIED SUCCESSFULLY


In [ ]:
from datasets import Dataset


def convert_dataframe(df):

    converted = []

    skipped = 0

    for _, row in df.iterrows():

        try:

            # REQUIRED FIELDS MUST EXIST

            if "image" not in row:
                skipped += 1
                continue

            if "question_lt" not in row:
                skipped += 1
                continue

            if "answer_lt" not in row:
                skipped += 1
                continue

            # REQUIRED FIELDS MUST NOT BE NaN

            if pd.isna(row["image"]):
                skipped += 1
                continue

            if pd.isna(row["question_lt"]):
                skipped += 1
                continue

            if pd.isna(row["answer_lt"]):
                skipped += 1
                continue

            # CONVERT TO CLEAN STRING

            image_name = str(row["image"]).strip()

            question = str(row["question_lt"]).strip()

            answer = str(row["answer_lt"]).strip()

            # EMPTY STRINGS NOT ALLOWED

            if image_name == "":
                skipped += 1
                continue

            if question == "":
                skipped += 1
                continue

            if answer == "":
                skipped += 1
                continue

            # IMAGE FILE MUST EXIST

            image_path = os.path.join(
                LOCAL_IMAGE_ROOT,
                row["dataset"],
                image_name
            )

            if not os.path.exists(image_path):

                print(f"MISSING IMAGE: {image_path}")

                skipped += 1
                continue

            # VERIFY IMAGE CAN ACTUALLY OPEN

            try:

                img = Image.open(image_path)
                img.verify()

            except Exception as e:

                print(f"CORRUPTED IMAGE: {image_path}")
                print(e)

                skipped += 1
                continue

            # BUILD CHAT MESSAGES

            messages = [

                {
                    "role": "user",

                    "content": [

                        {
                            "type": "image",
                            "image": image_path,
                        },

                        {
                            "type": "text",

                            "text": f"""
{GLOBAL_RULES}

Klausimas:
{question}

Atsakymas:
"""
                        },
                    ],
                },

                {
                    "role": "assistant",

                    "content": [

                        {
                            "type": "text",
                            "text": answer,
                        }
                    ],
                },
            ]

            converted.append({

                "image_path": image_path,

                "messages": messages,

                "question": question,

                "answer": answer,
            })

        except Exception as e:

            print("\nFAILED ROW:")
            print(e)

            skipped += 1

    print(f"\nVALID SAMPLES: {len(converted)}")
    print(f"SKIPPED SAMPLES: {skipped}")

    return Dataset.from_list(converted)

In [186]:
train_dataset = convert_dataframe(train_df)
val_dataset = convert_dataframe(val_df)


VALID SAMPLES: 19214
SKIPPED SAMPLES: 440

VALID SAMPLES: 958
SKIPPED SAMPLES: 42


In [ ]:

fast_eval_dataset = val_dataset.select(
    range(min(200, len(val_dataset)))
)

print("\nFAST EVAL SAMPLES:")
print(len(fast_eval_dataset))




FAST EVAL SAMPLES:
200


In [164]:
from unsloth import FastVisionModel

MODEL_NAME = "/home/vml/MedGemma-4B-IT-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(

    MODEL_NAME,

    load_in_4bit=False,

    use_gradient_checkpointing=True,

    attn_implementation="eager",
)

==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


In [165]:
model = FastVisionModel.get_peft_model(

    model,

    finetune_vision_layers=True,

    finetune_language_layers=True,

    finetune_attention_modules=True,

    finetune_mlp_modules=True,

    r=16,

    lora_alpha=16,

    lora_dropout=0,

    bias="none",

    random_state=3407,
)

In [166]:
class VisionCollator:

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):

        texts = []
        images = []

        for sample in batch:

            try:

                with Image.open(
                    sample["image_path"]
                ) as img:

                    image = img.convert("RGB")

                text = self.processor.apply_chat_template(
                    sample["messages"],
                    tokenize=False,
                    add_generation_prompt=False,
                )

                texts.append(text)

                images.append(image)

            except Exception as e:

                print("\nFAILED SAMPLE")
                print(sample["image_path"])
                print(e)

        assert len(texts) == len(images), (
            f"TEXTS={len(texts)} IMAGES={len(images)}"
        )

        encoding = self.processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        # Fix Gemma3 attention bug
        if (
            encoding["attention_mask"].shape[1]
            > encoding["input_ids"].shape[1]
        ):

            encoding["attention_mask"] = (
                encoding["attention_mask"][
                    :, :encoding["input_ids"].shape[1]
                ]
            )

        labels = encoding["input_ids"].clone()

        labels[
            labels == self.processor.tokenizer.pad_token_id
        ] = -100

        encoding["labels"] = labels

        return encoding

In [167]:
from transformers import Trainer, TrainingArguments

FastVisionModel.for_training(model)

training_args = TrainingArguments(

    output_dir=CHECKPOINT_DIR,

    report_to="tensorboard",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,  # EDIT depending on VRAM / dataset size

    learning_rate=2e-5,  # EDIT depending on convergence

    weight_decay=0.01,

    warmup_ratio=0.03,

    optim="adamw_torch",

    lr_scheduler_type="cosine",

    num_train_epochs=1,  # EDIT depending on dataset size

    logging_steps=10,

    eval_strategy="steps",

    eval_steps=600,  # EDIT depending on dataset size

    per_device_eval_batch_size=1,

    eval_accumulation_steps=1,

    save_strategy="steps",

    save_steps=600,  # EDIT depending on dataset size

    save_total_limit=3,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    remove_unused_columns=False,

    label_names=["labels"],

    dataloader_num_workers=8,  # EDIT depending on CPU

    dataloader_pin_memory=True,

    fp16=not torch.cuda.is_bf16_supported(),

    bf16=torch.cuda.is_bf16_supported(),

    seed=3407,
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=VisionCollator(tokenizer),
)



In [168]:
trainer_stats = trainer.train(
    # resume_from_checkpoint=True
)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,214 | Num Epochs = 1 | Total steps = 19,214
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 38,497,792 of 4,338,577,264 (0.89% trained)


Step,Training Loss,Validation Loss
600,2.383500,2.400837
1200,2.159200,2.183887
1800,2.175100,2.162087
2400,2.149600,2.153207
3000,2.143800,2.147663
3600,2.134300,2.145320
4200,2.137500,2.143346


KeyboardInterrupt: 

In [ ]:


BEST_CHECKPOINT = trainer.state.best_model_checkpoint
BEST_METRIC = trainer.state.best_metric

print("\n===================================")
print("BEST CHECKPOINT:")
print(BEST_CHECKPOINT)

print("\nBEST VALIDATION LOSS:")
print(BEST_METRIC)
print("===================================\n")


FastVisionModel.for_inference(model)



# SAVE FINAL BEST MODEL

model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("\nBEST MODEL SAVED:")
print(BEST_MODEL_DIR)


BEST CHECKPOINT:
/home/vml/notebooks/test/med-lt-project/medgemma/checkpoints/pathvqa_train_pathvqa_test_20260518_21/checkpoint-4200

BEST VALIDATION LOSS:
2.143346071243286


BEST MODEL SAVED:
/home/vml/notebooks/test/med-lt-project/medgemma/experiments/pathvqa_train_pathvqa_test_20260518_21/best_model


In [ ]:

# TEST INFERENCE


TEST_LIMIT = len(test_df) 

results = []

for row in test_df.head(TEST_LIMIT).itertuples(index=False):

    image_path = os.path.join(
        LOCAL_IMAGE_ROOT,
        row.dataset,
        row.image
    )

    try:

        with Image.open(image_path) as img:

            image = img.convert("RGB")

    except Exception as e:

        print("\nFAILED TEST IMAGE:")
        print(image_path)
        print(e)
        continue

    question = str(row.question_lt)
    correct_answer = str(row.answer_lt)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image"
                },
                {
                    "type": "text",
                    "text": f"""
{GLOBAL_RULES}

Klausimas:
{question}

Atsakymas:
"""
                }
            ]
        }
    ]

    # APPLY CHAT TEMPLATE

    input_text = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,
    )

    # BUILD INPUTS

    inputs = tokenizer(

        text=input_text,

        images=image,

        add_special_tokens=False,

        return_tensors="pt",
    )

    # MOVE TO CUDA

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    # FIX GEMMA3 ATTENTION BUG

    if (
        inputs["attention_mask"].shape[1]
        > inputs["input_ids"].shape[1]
    ):

        inputs["attention_mask"] = (
            inputs["attention_mask"][
                :, :inputs["input_ids"].shape[1]
            ]
        )

    # GENERATE

    with torch.inference_mode(), torch.autocast(

        device_type="cuda",

        dtype=torch.bfloat16
    ):

        outputs = model.generate(

            **inputs,

            max_new_tokens=16,  # EDIT if answers get cut off

            do_sample=False,

            eos_token_id=tokenizer.eos_token_id,

            pad_token_id=tokenizer.eos_token_id,
        )

    # REMOVE PROMPT TOKENS
  
    prompt_length = inputs["input_ids"].shape[1]

    generated = outputs[0][prompt_length:]

    # DECODE

    predicted_answer = tokenizer.decode(

        generated,

        skip_special_tokens=True,

    ).strip()

    result = {

        "dataset": row.dataset,

        "image": row.image,

        "question_lt": question,

        "correct_answer_lt": correct_answer,

        "predicted_answer_lt": predicted_answer,
    }

    results.append(result)

    print("\n================")
    print(result)




/home/vml/micromamba/envs/medgemma_clean/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



{'dataset': 'slake', 'image': 'test_xmlab102_source.jpg', 'question_lt': 'Koks būdas naudojamas šiam vaizdui fotografuoti?', 'correct_answer_lt': 'KT', 'predicted_answer_lt': 'radiografija'}

{'dataset': 'slake', 'image': 'test_xmlab102_source.jpg', 'question_lt': 'Kuriai kūno daliai priklauso šis vaizdas?', 'correct_answer_lt': 'Krūtinė', 'predicted_answer_lt': 'plaučiai'}

{'dataset': 'slake', 'image': 'test_xmlab102_source.jpg', 'question_lt': 'Koks yra pagrindinis organas paveikslėlyje?', 'correct_answer_lt': 'Plaučiai, nugaros smegenys', 'predicted_answer_lt': 'plaučiai'}

{'dataset': 'slake', 'image': 'test_xmlab102_source.jpg', 'question_lt': 'Koks yra didžiausias organas paveikslėlyje?', 'correct_answer_lt': 'Plaučiai', 'predicted_answer_lt': 'plaučiai'}

{'dataset': 'slake', 'image': 'test_xmlab102_source.jpg', 'question_lt': 'Ar paveikslėlyje yra kepenų?', 'correct_answer_lt': 'Ne', 'predicted_answer_lt': 'taip'}

{'dataset': 'slake', 'image': 'test_xmlab102_source.jpg', 'qu

In [188]:
predictions_df = pd.DataFrame(results)

predictions_csv_path = os.path.join(
    PREDICTIONS_DIR,
    "trained_pathvqa_others_test_predictions.csv"
)

predictions_df.to_csv(
    predictions_csv_path,
    index=False
)

print("\nPREDICTIONS SAVED:")
print(predictions_csv_path)


PREDICTIONS SAVED:
/home/vml/notebooks/test/med-lt-project/medgemma/experiments/pathvqa_train_other_test_20260519_11/predictions/trained_pathvqa_others_test_predictions.csv


In [ ]:
trainer.train(
    resume_from_checkpoint="/FULL/PATH/TO/CHECKPOINT"
)